## Universidad del Valle de Guatemala  
### Departamento de Ciencias de la Computacion  
### Inteligencia Artificial - Seccion 10
#### Nadissa Vela, 23764  
#### Cristian Tunchez, 231359

---

## Problemas de Satisfacción de Restricciones (CSP)

Desplegar 8 microservicios ($M_1$–$M_8$) en 3 servidores físicos ($S_1, S_2, S_3$) respetando:

1. **Restricción de Capacidad (Global):** Ningún servidor puede alojar más de 3 microservicios. Si se asignan 4, el peso de la asignación es 0.
2. **Restricciones de Anti-Afinidad (Binarias):** Las siguientes parejas no pueden estar en el mismo servidor:
   - $(M_1, M_2)$ — Base de datos primaria y su réplica
   - $(M_3, M_4)$
   - $(M_5, M_6)$
   - $(M_1, M_5)$

### Imports y Definición del Problema

In [9]:
import copy
from itertools import product

# Variables 
VARIABLES = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']

# Dominio de cada variable
DOMAIN = ['S1', 'S2', 'S3']

# Restricción de Capacidad
MAX_CAPACITY = 3

# Restricciones de Anti-Afinidad 
# Parejas que NO pueden estar en el mismo servidor
ANTI_AFFINITY = [
    ('M1', 'M2'),   # Base de datos primaria y su réplica
    ('M3', 'M4'),
    ('M5', 'M6'),
    ('M1', 'M5'),
]

print("Definición del Problema")
print(f"Variables   : {VARIABLES}")
print(f"Dominio     : {DOMAIN}")
print(f"Capacidad   : máx {MAX_CAPACITY} microservicios por servidor")
print(f"Anti-Afinidad: {ANTI_AFFINITY}")

Definición del Problema
Variables   : ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
Dominio     : ['S1', 'S2', 'S3']
Capacidad   : máx 3 microservicios por servidor
Anti-Afinidad: [('M1', 'M2'), ('M3', 'M4'), ('M5', 'M6'), ('M1', 'M5')]


### Funciones de Restricción (Constraints)

Estas funciones verifican si una asignación (completa o parcial) viola alguna restricción del problema.


In [10]:
def check_anti_affinity(assignment):
    """
    Retorna True si NINGUNA pareja de anti-afinidad comparte servidor.
    Solo evalúa pares donde AMBAS variables están ya asignadas.
    """
    for (mi, mj) in ANTI_AFFINITY:
        if mi in assignment and mj in assignment:
            if assignment[mi] == assignment[mj]:
                return False
    return True


def check_capacity(assignment):
    """
    Retorna True si ningún servidor supera MAX_CAPACITY microservicios.
    """
    server_count = {}
    for server in assignment.values():
        server_count[server] = server_count.get(server, 0) + 1
    return all(count <= MAX_CAPACITY for count in server_count.values())


def compute_weight(assignment):
    """
    Peso de una asignación COMPLETA: 1 si es válida, 0 si viola alguna restricción.
    """
    if len(assignment) < len(VARIABLES):
        raise ValueError("compute_weight requiere una asignación completa.")
    if check_anti_affinity(assignment) and check_capacity(assignment):
        return 1
    return 0


# Prueba rápida
test_ok  = {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2',
            'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
test_bad = {'M1': 'S1', 'M2': 'S1', 'M3': 'S2', 'M4': 'S3',  # M1==M2 violación
            'M5': 'S2', 'M6': 'S3', 'M7': 'S1', 'M8': 'S2'}

print(f"test_ok  anti_affinity={check_anti_affinity(test_ok)},  capacity={check_capacity(test_ok)},  weight={compute_weight(test_ok)}")
print(f"test_badanti_affinity={check_anti_affinity(test_bad)}, capacity={check_capacity(test_bad)}, weight={compute_weight(test_bad)}")

test_ok  anti_affinity=True,  capacity=True,  weight=1
test_badanti_affinity=False, capacity=True, weight=0


### Función Heurística para Beam Search

Para el Beam Search necesitamos puntuar asignaciones parciales y elegir las mejores K. La heurística penaliza violaciones usando la fórmula:

$$w_{heuristic}(x) = 1 - \frac{\text{violaciones actuales}}{\text{total de restricciones}}$$


In [11]:
def count_violations(assignment):
    """
    Cuenta el número total de violaciones en una asignación (parcial o completa).
      - Anti-afinidad: +1 por cada pareja prohibida que comparte servidor.
      - Capacidad: +1 por cada microservicio que excede MAX_CAPACITY en un servidor.
    """
    violations = 0

    # Anti-afinidad (solo evalúa pares donde ambas vars están asignadas)
    for (mi, mj) in ANTI_AFFINITY:
        if mi in assignment and mj in assignment:
            if assignment[mi] == assignment[mj]:
                violations += 1

    # Capacidad
    server_count = {}
    for server in assignment.values():
        server_count[server] = server_count.get(server, 0) + 1
    for count in server_count.values():
        if count > MAX_CAPACITY:
            violations += (count - MAX_CAPACITY)   # exceso sobre el límite

    return violations


# Normalizador: número total de restricciones del problema
TOTAL_CONSTRAINTS = len(ANTI_AFFINITY) + len(VARIABLES)


def heuristic_weight(assignment):
    """
    Heurística normalizada para Beam Search.
    
    w(x) = 1 - violaciones / total_restricciones
    
    Rango: [0, 1]. Mayor es mejor (menos violaciones).
    """
    v = count_violations(assignment)
    return 1.0 - v / TOTAL_CONSTRAINTS


# Prueba
print(f"Sin violaciones  {heuristic_weight({'M1': 'S1', 'M2': 'S2'}):.4f}  (esperado ~1.0)")
print(f"Una violación    {heuristic_weight({'M1': 'S1', 'M2': 'S1'}):.4f}  (esperado <1.0)")
print(f"TOTAL_CONSTRAINTS = {TOTAL_CONSTRAINTS}")

Sin violaciones  1.0000  (esperado ~1.0)
Una violación    0.9167  (esperado <1.0)
TOTAL_CONSTRAINTS = 12
